In [0]:
%sql
show catalogs

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS gizmobox
MANAGED LOCATION 'abfss://gizmobox@resourcedatabricks.dfs.core.windows.net/'
COMMENT 'this is a lab for gizmobox lab'


In [0]:
%sql
use catalog gizmobox;

In [0]:
%sql
select current_catalog();
    

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS landing
MANAGED LOCATION 'abfss://gizmobox@resourcedatabricks.dfs.core.windows.net/landing';

CREATE SCHEMA IF NOT EXISTS bronze
MANAGED LOCATION 'abfss://gizmobox@resourcedatabricks.dfs.core.windows.net/bronze';

CREATE SCHEMA IF NOT EXISTS silver
MANAGED LOCATION 'abfss://gizmobox@resourcedatabricks.dfs.core.windows.net/silver';

CREATE SCHEMA IF NOT EXISTS gold
MANAGED LOCATION 'abfss://gizmobox@resourcedatabricks.dfs.core.windows.net/gold';

In [0]:
%sql
use schema landing;
CREATE EXTERNAL VOLUME IF NOT EXISTS operational_data 
LOCATION 'abfss://gizmobox@resourcedatabricks.dfs.core.windows.net/landing/operational_data/';

In [0]:
%fs ls /Volumes/gizmobox/landing/operational_data/customers/

In [0]:
%sql
select * from json.`dbfs:/Volumes/gizmobox/landing/operational_data/customers/customers_2024_10.json`;

In [0]:
%sql
select _metadata.file_path as file_name, 
      * 
from json.`dbfs:/Volumes/gizmobox/landing/operational_data/customers/`;

In [0]:
%sql
CREATE OR REPLACE VIEW gizmobox.bronze.customers
AS
select _metadata.file_path as file_path, 
      * 
from json.`dbfs:/Volumes/gizmobox/landing/operational_data/customers/`;

In [0]:
%sql select * from gizmobox.bronze.customers

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW tmp_customers
AS
select _metadata.file_path as file_path, 
      * 
from json.`dbfs:/Volumes/gizmobox/landing/operational_data/customers/`;

In [0]:
%sql
select * from tmp_customers;

In [0]:
%sql
--it is only supported when creating clusters, not when using serverless, because viwes live in the cluster
CREATE OR REPLACE GLOBAL TEMPORARY VIEW gtv_customers
AS
select _metadata.file_path as file_path, 
      * 
from json.`dbfs:/Volumes/gizmobox/landing/operational_data/customers/`;

In [0]:
%sql
select * from json.`dbfs:/Volumes/gizmobox/landing/operational_data/orders`;

In [0]:
%sql
CREATE OR REPLACE VIEW gizmobox.bronze.v_orders
AS
select * from TEXT.`dbfs:/Volumes/gizmobox/landing/operational_data/orders`;

In [0]:
%sql
select * from gizmobox.bronze.v_orders;

In [0]:
%sql
CREATE OR REPLACE VIEW gizmobox.bronze.v_memberships
AS
select * from binaryFile.`dbfs:/Volumes/gizmobox/landing/operational_data/memberships/*/*.png`;

In [0]:
%sql
select * from gizmobox.bronze.v_memberships;

In [0]:
%sql
CREATE OR REPLACE VIEW gizmobox.bronze.v_addresses
AS
select * from read_files('dbfs:/Volumes/gizmobox/landing/operational_data/addresses', format => 'csv', delimiter => '\t', header => true)

In [0]:
%sql
select * from gizmobox.bronze.v_addresses;

In [0]:
%fs ls 'abfss://gizmobox@resourcedatabricks.dfs.core.windows.net/landing/external_data/payments'

In [0]:
%sql
CREATE TABLE IF NOT EXISTS gizmobox.bronze.payments(
  payment_id INT,
  order_id INT,
  payment_timestamp TIMESTAMP,
  payment_status Integer,
  payment_method STRING
  )
  USING csv
  OPTIONS (
    header = "true",
    delimiter = ","
  )
  LOCATION 'abfss://gizmobox@resourcedatabricks.dfs.core.windows.net/landing/external_data/payments';
    
select * from gizmobox.bronze.payments

In [0]:
%sql
describe extended gizmobox.bronze.payments;